## Imports and Setup

In [ ]:
#Core libraries
import pandas as pd
import numpy as np
# XGBoost
from xgboost import XGBRegressor
# Scikit-learn utilities
from sklearn.model_selection import train_test_split, GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error
# Model persistence
import joblib
# Plotting
import matplotlib.pyplot as plt
# Reproducibility
RANDOM_STATE = 42

import json

## Load Featured Dataset

In [ ]:
df = pd.read_csv(
    "../../data/processed/feature_engineered_unscaled.csv",
    parse_dates=[0],
    index_col=0
)

df = df.sort_index()
df.head()

## Select Features and Target

In [ ]:
target = "demand"

features = [
    "lag_1",
    "lag_48",
    "rolling_mean_48",
    "hour",
    "day_of_week",
    "month"
]

df = df.dropna()

X = df[features]
y = df[target]

## Train-Test Split

In [ ]:
# Fixed split used across ALL models (ARIMA, XGBoost, LSTM)
SPLIT_RATIO = 0.8

split_idx = int(len(df) * SPLIT_RATIO)
SPLIT_TIMESTAMP = df.index[split_idx]

print(f"Official train-test split timestamp: {SPLIT_TIMESTAMP}")

## Apply the Split

In [ ]:
train_df = df.loc[:SPLIT_TIMESTAMP]
test_df = df.loc[SPLIT_TIMESTAMP:]

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

## Hyperparamter Tuning with GridSearchCV

In [ ]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42
)

param_grid = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1]
}

tscv = TimeSeriesSplit(n_splits=3)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_mean_squared_error",
    verbose=1
)

grid_search.fit(X_train, y_train)

## Evaluate XGBoost

In [ ]:
best_xgb = grid_search.best_estimator_

preds = best_xgb.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, preds))
mae = mean_absolute_error(y_test, preds)

rmse, mae

In [ ]:
# Save metrics to a JSON file
xgb_results = {
    'model_name': 'XGBoost',
    'rmse': float(rmse),
    'mae': float(mae)
}

with open('../results/xgb_metrics.json', 'w') as f:
    json.dump(xgb_results, f)
    
# Optional: Save predictions for the evaluation plot later
np.save('../results/xgb_preds.npy', preds[:100]) # Save first 100 for plotting

# Save the Model

In [ ]:
joblib.dump(
    best_xgb,
    "xgb_model.pkl"
)